# From Regular Expressions to <span style="font-variant:small-caps;">Nfa</span>s

This notebook shows how a given regular expression $r$ can be transformed into an equivalent finite state machine. 
It implements the theory that is outlined in section 4.4. of the 
lecture notes.

In [ ]:
#use "../lib/nb.ml";;
run_notebook "Types.ipynb"

The type `regexp` describes the *parse tree* of a regular expression.  This will be the *input* of the program we
develop in this notebook.  Note that `regexp` is a *recursive* type.  It is defined in the notebook `Types.ipynb`.

In [ ]:
#show_type regexp

We will represent the states of the `NFA` as integers.

In [ ]:
type state = int

The type `state trans_rel` denotes the transition relation of a non-deterministic finite automaton.
The type `state nfa` denotes a non-deterministic finite automaton whose states are integers.  Its elements are records with the five fields
`states`, `sigma`, `delta`, `start`, and `accepting`, which correspond to the 5-tuples
$$ \langle Q, \Sigma, \delta, q_0, A \rangle $$
where
1. $Q$ is the set of states,
2. $\Sigma$ is the alphabet,
3. $\delta: Q \times (\Sigma \cup \{ \varepsilon \}) \rightarrow 2^Q$ is the transition relation,
4. $q_0 \in Q$ is the start state, and
5. $A \subseteq Q$ is the set of accepting states

In [ ]:
#show_type trans_rel;;
#show_type nfa

In Python, the class `RegExp2NFA` administers two member variables:
- `Sigma` is the <em style="color:blue">alphabet</em>, i.e. the set of characters used.
- `StateCount` is a counter that is needed to create <em style="color:blue">unique</em> state names.

In OCaml, we use the record type `regExp2NFA` with the fields `alphabet` and `stateCount`.  The field `stateCount`
is declared as `mutable`, so it can be changed.  The function `regExp2NFA sigma` creates a new converter and
corresponds to the constructor call `RegExp2NFA(Sigma)`.  The *methods* of the Python class become ordinary
functions that take the converter `self` as their first argument.

The Python class contains stubs of the methods, which are implemented later and then attached to the class.
In OCaml, a function can only call functions that have already been defined.  Therefore, we create *forward
references* for these functions: a reference is a mutable box, and `!toNFA_ref` is the function that is
currently stored in the box `toNFA_ref`.  Initially, every box contains the function `not_yet_defined`.  Once a
function has been implemented, it is stored in its box via the operator `:=`.  This corresponds exactly to the
Python statements of the form `RegExp2NFA.toNFA = toNFA`.  The types of the references play the role of the
signatures of the stubs.

In [ ]:
type regExp2NFA = { alphabet : char FSet.t; mutable stateCount : int }

let regExp2NFA (sigma : char FSet.t) : regExp2NFA = { alphabet = sigma; stateCount = 0 }

let toNFA_ref         : (regExp2NFA -> regexp -> state nfa) ref                   = ref not_yet_defined
let genEmptyNFA_ref   : (regExp2NFA -> state nfa) ref                             = ref not_yet_defined
let genEpsilonNFA_ref : (regExp2NFA -> state nfa) ref                             = ref not_yet_defined
let genCharNFA_ref    : (regExp2NFA -> char -> state nfa) ref                     = ref not_yet_defined
let catenate_ref      : (regExp2NFA -> state nfa -> state nfa -> state nfa) ref   = ref not_yet_defined
let disjunction_ref   : (regExp2NFA -> state nfa -> state nfa -> state nfa) ref   = ref not_yet_defined
let kleene_ref        : (regExp2NFA -> state nfa -> state nfa) ref                = ref not_yet_defined
let getNewState_ref   : (regExp2NFA -> state) ref                                 = ref not_yet_defined

The function `toNFA` takes a converter `self` and a regular expression `r` and returns a finite state machine 
that accepts the same language as described by `r`.  The regular expression is represented as a value of the type `regexp`:
- The regular expression $\emptyset$ is represented as `Empty`.
- The regular expression $\varepsilon$ is represented as `Epsilon`.
- The regular expression $c$ that matches the character $c$ is represented by `Char c`.
- The regular expression $r_1 \cdot r_2$  is represented by `Product (r1, r2)`.
- The regular expression $r_1 + r_2$  is represented by `Sum (r1, r2)`.
- The regular expression $r^*$  is represented by `Star r`.

**Type safety:** The Python version needs a final case `case _: raise ValueError(...)` for values that are not
regular expressions, and the annotation `# type: ignore` to silence the type checker.  In OCaml, the compiler
checks that the `match` handles every constructor of the type `regexp`, so no such case is needed.

OCaml does not specify the order in which the arguments of a function are evaluated, and in practice they are
evaluated from right to left.  Since `toNFA` changes the counter `stateCount`, the order matters: it determines
the numbers of the states.  In order to get the same numbering as in Python, we use `let` to convert `r1`
before `r2`.

**Function `toNFA self r`**
- *Input:* `self` is a converter of type `regExp2NFA` and `r` is a regular expression.
- *Output:* An <span style="font-variant:small-caps;">Nfa</span> that accepts the language described by `r`.

In [ ]:
let rec toNFA (self : regExp2NFA) (r : regexp) : state nfa =
  match r with
  | Empty -> !genEmptyNFA_ref self
  | Epsilon -> !genEpsilonNFA_ref self
  | Char c -> !genCharNFA_ref self c
  | Product (r1, r2) ->
      let f1 = toNFA self r1 in
      let f2 = toNFA self r2 in
      !catenate_ref self f1 f2
  | Sum (r1, r2) ->
      let f1 = toNFA self r1 in
      let f2 = toNFA self r2 in
      !disjunction_ref self f1 f2
  | Star r1 -> !kleene_ref self (toNFA self r1)

In [ ]:
toNFA_ref := toNFA

The <span style="font-variant:small-caps;">Nfa</span> `genEmptyNFA self` is defined as
$$\bigl\langle \{ q_0, q_1 \}, \Sigma, \{\}, q_0, \{ q_1 \} \bigr\rangle. $$
Note that this <span style="font-variant:small-caps;">Nfa</span> has no transitions at all.
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:

![Nfa recognizing the empty set](./aLeer.jpg)

**Function `genEmptyNFA self`**
- *Input:* `self` is a converter of type `regExp2NFA`.
- *Output:* An <span style="font-variant:small-caps;">Nfa</span> that accepts the empty language.

In [ ]:
let genEmptyNFA (self : regExp2NFA) : state nfa =
  let q0 = !getNewState_ref self in
  let q1 = !getNewState_ref self in
  { states = FSet.of_list [ q0; q1 ]; sigma = self.alphabet; delta = Dict.empty; start = q0; accepting = FSet.singleton q1 }

In [ ]:
genEmptyNFA_ref := genEmptyNFA

The <span style="font-variant:small-caps;">Nfa</span> `genEpsilonNFA` is defined as
$$  \bigl\langle \{ q_0, q_1 \}, \Sigma, 
                          \bigl\{ \langle q_0, \varepsilon\rangle \mapsto \{q_1\} \bigr\}, q_0, \{ q_1 \} \bigr\rangle.
$$
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:

![Nfa recognizing the empty string](./aEpsilon.jpg)

**Function `genEpsilonNFA self`**
- *Input:* `self` is a converter of type `regExp2NFA`.
- *Output:* An <span style="font-variant:small-caps;">Nfa</span> that accepts only the empty string.

In [ ]:
let genEpsilonNFA (self : regExp2NFA) : state nfa =
  let q0 = !getNewState_ref self in
  let q1 = !getNewState_ref self in
  let delta = Dict.of_list [ ((q0, Eps), FSet.singleton q1) ] in
  { states = FSet.of_list [ q0; q1 ]; sigma = self.alphabet; delta; start = q0; accepting = FSet.singleton q1 }

In [ ]:
genEpsilonNFA_ref := genEpsilonNFA

For a letter $c \in \Sigma$ the <span style="font-variant:small-caps;">NFA</span> `genCharNFA`$(c)$ is defined as 
$$ A(c) = 
   \bigl\langle \{ q_0, q_1 \}, \Sigma, 
   \bigl\{ \langle q_0, c \rangle \mapsto \{q_1\}\bigr\}, q_0, \{ q_1 \} \bigr\rangle.
$$
Graphically, this <span style="font-variant:small-caps;">NFA</span> looks as follows:

![NFA recognizing the character c](./aChar.jpg)

**Function `genCharNFA self c`**
- *Input:* `self` is a converter of type `regExp2NFA` and `c` is a character.
- *Output:* An <span style="font-variant:small-caps;">Nfa</span> that accepts only the string `c`.

In [ ]:
let genCharNFA (self : regExp2NFA) (c : char) : state nfa =
  let q0 = !getNewState_ref self in
  let q1 = !getNewState_ref self in
  let delta = Dict.of_list [ ((q0, Sym c), FSet.singleton q1) ] in
  { states = FSet.of_list [ q0; q1 ]; sigma = self.alphabet; delta; start = q0; accepting = FSet.singleton q1 }

In [ ]:
genCharNFA_ref := genCharNFA

Given two <span style="font-variant:small-caps;">Nfa</span>s `f1` and `f2`, the function `catenate self f1 f2` 
creates an <span style="font-variant:small-caps;">Nfa</span> that recognizes a string $s$ if it can be written 
in the form
$$ s = s_1s_2 $$
and $s_1$ is recognized by `f1` and $s_2$ is recognized by `f2`. 

Assume that $f_1$ and $f_2$ have the following form:
- $f_1 = \langle Q_1, \Sigma, \delta_1, q_1, \{ q_2 \}\rangle$,
- $f_2 = \langle Q_2, \Sigma, \delta_2, q_3, \{ q_4 \}\rangle$,
- $Q_1 \cap Q_2 = \{\}$.
 
Then $\texttt{catenate}(f_1, f_2)$ is defined as:
$$  \bigl\langle Q_1 \cup Q_2, \Sigma, 
   \bigl\{ \langle q_2,\varepsilon\rangle  \mapsto \{q_3\} \bigr\} 
         \cup \delta_1 \cup \delta_2, q_1, \{ q_4 \} \bigr\rangle.
$$
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:

![Nfa recognizing the concatenation of two languages](./aConcat.jpg)

**Function `catenate self f1 f2`**
- *Input:* `self` is a converter of type `regExp2NFA` and `f1` and `f2` are <span style="font-variant:small-caps;">Nfa</span>s with disjoint sets of states.
- *Output:* An <span style="font-variant:small-caps;">Nfa</span> that accepts the concatenation of the languages of `f1` and `f2`.

The Python statement `q2, = A1` extracts the only element of the set `A1`.  In OCaml, the function `the_element`
does this.  The pattern `{ states = m1; ... }` extracts the fields of the record `f1`.  Its type annotation
`state nfa` tells the compiler that `f1` is an <span style="font-variant:small-caps;">Nfa</span>.

In [ ]:
let the_element (s : 'a FSet.t) : 'a =
  match FSet.to_list s with
  | [ x ] -> x
  | _ -> invalid_arg "the_element: the set does not have exactly one element"

let catenate (self : regExp2NFA) (f1 : state nfa) (f2 : state nfa) : state nfa =
  let ({ states = m1; sigma; delta = delta1; start = q1; accepting = a1 } : state nfa) = f1 in
  let ({ states = m2; delta = delta2; start = q3; accepting = a2; _ } : state nfa) = f2 in
  let q2 = the_element a1 in
  let delta = Dict.add (q2, Eps) (FSet.singleton q3) (Dict.union delta1 delta2) in
  { states = FSet.union m1 m2; sigma; delta; start = q1; accepting = a2 }

In [ ]:
catenate_ref := catenate

Given two <span style="font-variant:small-caps;">Nfa</span>s `f1` and `f2`, the function `disjunction self f1 f2` 
creates an <span style="font-variant:small-caps;">Nfa</span> that recognizes a string $s$ if it is either 
is recognized by `f1` or by `f2`. 

Assume again that the states of 
$f_1$ and $f_2$ are different and that $f_1$ and $f_2$ have the following form:
- $f_1 = \langle Q_1, \Sigma, \delta_1, q_1, \{ q_3 \}\rangle$,
- $f_2 = \langle Q_2, \Sigma, \delta_2, q_2, \{ q_4 \}\rangle$,
- $Q_1 \cap Q_2 = \{\}$.

Then $\texttt{disjunction}(f_1, f_2)$ is defined as follows:
$$ \bigl\langle \{ q_0, q_5 \} \cup Q_1 \cup Q_2, \Sigma, 
                \bigl\{ \langle q_0,\varepsilon\rangle \mapsto \{q_1, q_2\},
                   \langle q_3,\varepsilon\rangle \mapsto \{q_5\}, 
                   \langle q_4,\varepsilon\rangle \mapsto \{q_5\} \bigr\} 
                   \cup \delta_1 \cup \delta_2, q_0, \{ q_5 \} \bigr\rangle
$$
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:
![Nfa recognizing the disjunction](./aPlus.jpg)

**Function `disjunction self f1 f2`**
- *Input:* `self` is a converter of type `regExp2NFA` and `f1` and `f2` are <span style="font-variant:small-caps;">Nfa</span>s with disjoint sets of states.
- *Output:* An <span style="font-variant:small-caps;">Nfa</span> that accepts the union of the languages of `f1` and `f2`.

In [ ]:
let disjunction (self : regExp2NFA) (f1 : state nfa) (f2 : state nfa) : state nfa =
  let ({ states = q1s; sigma; delta = delta1; start = q1; accepting = a1 } : state nfa) = f1 in
  let ({ states = q2s; delta = delta2; start = q2; accepting = a2; _ } : state nfa) = f2 in
  let q3 = the_element a1 in
  let q4 = the_element a2 in
  let q0 = !getNewState_ref self in
  let q5 = !getNewState_ref self in
  let delta =
    Dict.union delta1 delta2
    |> Dict.add (q0, Eps) (FSet.of_list [ q1; q2 ])
    |> Dict.add (q3, Eps) (FSet.singleton q5)
    |> Dict.add (q4, Eps) (FSet.singleton q5)
  in
  { states = FSet.union (FSet.of_list [ q0; q5 ]) (FSet.union q1s q2s); sigma; delta; start = q0;
    accepting = FSet.singleton q5 }

In [ ]:
disjunction_ref := disjunction

Given an <span style="font-variant:small-caps;">Nfa</span> `f`, the function `kleene self f` 
creates an <span style="font-variant:small-caps;">Nfa</span> that recognizes a string $s$ if it can be written as
$$ s = s_1 s_2 \cdots s_n $$
and all $s_i$ are recognized by `f`.  Note that $n$ might be $0$. 

If `f` is defined as
$$ f = \langle Q, \Sigma, \delta, q_1, \{ q_2 \} \rangle,
$$
then  `kleene self f` is defined as follows:
$$ \bigl\langle \{ q_0, q_3 \} \cup Q, \Sigma, 
                \bigl\{ \langle q_0,\varepsilon\rangle \mapsto \{q_1, q_3\},  
                        \langle q_2,\varepsilon\rangle \mapsto \{q_1, q_3\} \bigr\} 
                \cup \delta, q_0, \{ q_3 \} \bigr\rangle.
$$
Graphically, this <span style="font-variant:small-caps;">Nfa</span> looks as follows:
![Nfa recognizing the Kleene star](./aStar.jpg)

The Python version has to copy the dictionary `delta0` before it is changed, since otherwise the
<span style="font-variant:small-caps;">Nfa</span> `f` would be changed, too.  In OCaml, dictionaries are
*immutable*: `Dict.add` returns a new dictionary and leaves `delta0` unchanged, so there is no need to copy.

**Function `kleene self f`**
- *Input:* `self` is a converter of type `regExp2NFA` and `f` is an <span style="font-variant:small-caps;">Nfa</span>.
- *Output:* An <span style="font-variant:small-caps;">Nfa</span> that accepts the Kleene closure of the language of `f`.

In [ ]:
let kleene (self : regExp2NFA) (f : state nfa) : state nfa =
  let ({ states = q; sigma; delta = delta0; start = q1; accepting = a } : state nfa) = f in
  let q2 = the_element a in
  let q0 = !getNewState_ref self in
  let q3 = !getNewState_ref self in
  let delta =
    delta0
    |> Dict.add (q0, Eps) (FSet.of_list [ q1; q3 ])
    |> Dict.add (q2, Eps) (FSet.of_list [ q1; q3 ])
  in
  { states = FSet.union (FSet.of_list [ q0; q3 ]) q; sigma; delta; start = q0; accepting = FSet.singleton q3 }

In [ ]:
kleene_ref := kleene

The auxiliary function `getNewState` returns a new number that has not yet been used as a state.

**Function `getNewState self`**
- *Input:* `self` is a converter of type `regExp2NFA`.
- *Output:* A number that has not yet been used as a state.

In [ ]:
let getNewState (self : regExp2NFA) : state =
  self.stateCount <- self.stateCount + 1;
  self.stateCount

In [ ]:
getNewState_ref := getNewState

The notebook `04-Test-Regexp-2-NFA`can be used to test the functions implemented in this notebook.